# Try Embedding Search

This notebook tests the semantic search capability of the Navigational Knowledge Graph (NKG).
It converts a user's natural language intent into a vector using the Ollama Proxy and finds the most relevant UI elements in Neo4j using the vector index.

In [2]:
import os
import json
import requests
from dotenv import load_dotenv
from neo4j import GraphDatabase

# Load variables from the .env file in this directory
load_dotenv()

# --- Configuration ---
NEO4J_URI          = os.getenv("NEO4J_URI")
NEO4J_USER         = os.getenv("NEO4J_USER")
NEO4J_PASSWORD     = os.getenv("NEO4J_PASSWORD")
OLLAMA_PROXY_URL   = os.getenv("OLLAMA_PROXY_URL")
OLLAMA_PROXY_TOKEN = os.getenv("OLLAMA_PROXY_TOKEN")

EMBEDDING_MODEL    = "qwen3-embedding:8b"

QUERY_INSTRUCTION = (
    "Given a user's navigation intent or action description, find the most relevant "
    "UI element in the web application that the user wants to interact with."
)

print(f"Proxy URL: {OLLAMA_PROXY_URL}")

# --- Connect to Neo4j ---
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("✓ Connected to Neo4j")

Proxy URL: https://youtube.com
✓ Connected to Neo4j


In [3]:
def get_query_embedding(text: str) -> list[float]:
    """
    Calls the Ollama Proxy to get the embedding vector for the query.
    Uses the asymmetric QUERY_INSTRUCTION for qwen3-embedding.
    """
    # Format the instruction prompt exactly as the model expects
    prompt = f"Instruct: {QUERY_INSTRUCTION}\nQuery: {text}"
    
    headers = {"X-API-Token": OLLAMA_PROXY_TOKEN}
    payload = {
        "model": EMBEDDING_MODEL,
        "prompt": prompt
    }
    
    response = requests.post(f"{OLLAMA_PROXY_URL.rstrip('/')}/api/embeddings", json=payload, headers=headers)
    if response.status_code != 200:
        print(f"Error: {response.status_code} - {response.text}")
    response.raise_for_status()
    return response.json()["embedding"]

In [4]:
def search_elements(query_text: str, top_n: int = 5):
    """
    Searches the Neo4j vector index for the top N most similar elements.
    """
    query_vector = get_query_embedding(query_text)
    
    cypher = """
    CALL db.index.vector.queryNodes('element_embedding_idx', $top_n, $vector)
    YIELD node, score
    RETURN 
        node.nkg_id AS nkg_id,
        node.type AS type,
        node.desc AS desc,
        node.selector AS selector,
        score
    """
    
    with driver.session() as session:
        result = session.run(cypher, vector=query_vector, top_n=top_n)
        return result.data()

## Test the Semantic Search

In [6]:
USER_QUERY = "beli paket prime"

print(f"Searching for: '{USER_QUERY}'...\n")
results = search_elements(USER_QUERY, 15)

if not results:
    print("No matching elements found. Make sure the index is populated!")
else:
    print(f"{'Score':<8} | {'NKG ID':<50} | {'Description'}")
    print("-" * 110)
    for r in results:
        print(f"{r['score']:.4f}   | {r['nkg_id']:<50} | {r['desc']}")

Searching for: 'beli paket prime'...

Score    | NKG ID                                             | Description
--------------------------------------------------------------------------------------------------------------
0.8561   | /customer/price_list/package_container3            | Kartu detail Paket Prime
0.8489   | /customer/price_list/packageName3                  | Nama paket: Paket Prime
0.8468   | /customer/price_list/package_detail_3              | Detail paket harga Prime, menampilkan harga Rp 464.384 dan daftar fitur yang didapatkan
0.8368   | /customer/setting/billing/info/info__info_icon_prime | Ikon informasi untuk melihat detail paket Prime
0.8328   | /customer/price_list/badgePackage3                 | Badge status 'Berlangganan' pada Paket Prime
0.8316   | /customer/price_list/period_3                      | Periode masa berlaku untuk paket Prime
0.8233   | /customer/price_list/priceValue_3                  | Nilai numerik harga untuk paket Prime (hidden)
0.8190   

In [ ]:
driver.close()